# Train a custom player/goalkeeper/referee detector (YOLO26)

Before running anything:
1. In the right-hand panel of this Kaggle notebook, open **Settings**.
2. Set **Accelerator** to **GPU T4 x2** (or any GPU option).
3. Turn **Internet** ON (needed to install packages and download the dataset).

This trains on the public `football-players-detection` dataset (player, goalkeeper,
referee, ball -- ~660 labeled broadcast-football images), which fixes the two
biggest accuracy problems with the generic pretrained model: it can't tell a
referee from a player, and it wasn't trained on football footage at all.

In [ ]:
!pip install -q -U ultralytics roboflow

## Step 1: Download the dataset

Go to this page and sign in (free account):
https://universe.roboflow.com/roboflow-jvuqo/football-players-detection-3zvbc

Click **Download Dataset** on that page, choose format **YOLO26** (or **YOLOv11**
if YOLO26 isn't listed as a format option yet -- the label format is the same,
only the model architecture below needs to say YOLO26), and select
"Show download code" instead of downloading a zip. Roboflow will generate a code
snippet with YOUR api key and the correct dataset version already filled in --
paste that snippet into the cell below, replacing the placeholder.

In [ ]:
# PASTE the code snippet Roboflow generated for you here, replacing
# everything below. Use EXACTLY what Roboflow shows you -- especially
# the string inside version.download(...), e.g. "yolo26" or "yolov11"
# depending what format you picked. It will look like this (with YOUR
# real api_key and version number, not these placeholder values):
#
# from roboflow import Roboflow
# rf = Roboflow(api_key="YOUR_API_KEY")
# project = rf.workspace("roboflow-jvuqo").project("football-players-detection-3zvbc")
# version = project.version(12)
# dataset = version.download("yolo26")

from roboflow import Roboflow

rf = Roboflow(api_key="PASTE_YOUR_API_KEY_HERE")
project = rf.workspace("roboflow-jvuqo").project("football-players-detection-3zvbc")
version = project.version(12)  # replace with whatever version number Roboflow gave you
dataset = version.download("yolo26")  # replace with whatever format string Roboflow gave you

print("Dataset downloaded to:", dataset.location)

In [ ]:
# Sanity check: confirm the classes match what we expect
# (player, goalkeeper, referee, ball -- order may vary)
import yaml

data_yaml_path = f"{dataset.location}/data.yaml"

with open(data_yaml_path) as f:
    data_yaml = yaml.safe_load(f)

print("Classes:", data_yaml["names"])
print("Number of classes:", data_yaml["nc"])

## Step 2: Train

Starting from `yolo26s.pt` (small, pretrained on general objects) and
fine-tuning on the football dataset. This is much faster and more accurate
than training from scratch.

50 epochs on ~660 images with a T4 GPU should take roughly 30-60 minutes.
If Kaggle's session times out or you want to stop early, YOLO saves
checkpoints as it goes -- `last.pt` in the run folder can resume.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo26s.pt")

results = model.train(
    data=data_yaml_path,
    epochs=50,
    imgsz=960,          # broadcast football has small/far-away players -- higher
                          # res than the yolo default (640) helps catch them
    batch=16,
    patience=15,          # stop early if validation stops improving
    project="football_training",
    name="player_detector",
    device=0               # use the GPU
)

## Step 3: Validate

Check the metrics before you trust this model. `mAP50` above ~0.7 is a solid
result for this dataset size; if it's much lower, something likely went wrong
(check the class list printed above matches what you expect).

In [ ]:
metrics = model.val()
print(metrics.box.map50)   # mAP at IoU 0.5 -- higher is better, max 1.0

## Step 4: Get your trained weights

This copies the best checkpoint to `/kaggle/working/` so it shows up in the
notebook's **Output** panel (right sidebar) where you can download it.

In [ ]:
import shutil

best_weights = f"{results.save_dir}/weights/best.pt"
output_path = "/kaggle/working/yolo26_player.pt"

shutil.copy(best_weights, output_path)

print(f"Saved to {output_path}")
print("Download it from the 'Output' panel on the right side of this notebook.")
print()
print("Classes this model detects, in order:", data_yaml["names"])
print("Remember these -- you'll need them to update PLAYER_CLASSES in config.py")